# Lucas Phase 1.1: IV + FE Evaluation

This notebook fills the Phase 1 gap by adding controls and IV candidates, running second-stage models, and evaluating what holds vs what fails.

## Scope
- Use newly created files:
  - `data/phase1_controls.csv`
  - `data/phase1_instruments.csv`
- Run FE + controls.
- Run IV second-stage with two candidate strategies.
- Produce a compact evaluation and decision summary.


## Plan for Phase 1.1

1. Load base panel + new controls + new instrument file.
2. Run data quality and overlap checks.
3. Estimate FE + controls baselines.
4. Diagnose first stage for IV candidates.
5. Run 2SLS second stage:
   - Spec A: Two-way FE + lagged money-growth instrument.
   - Spec B: Country FE + external Bartik-style instrument.
6. Compare outputs and evaluate model credibility.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path('/Users/stevenchung/Desktop/P12B_File/New project_')
BASE_DATA = PROJECT_ROOT / 'macro_growth_merged.csv'
CTRL_DATA = PROJECT_ROOT / 'data/phase1_controls.csv'
IV_DATA = PROJECT_ROOT / 'data/phase1_instruments.csv'

OUT_ROOT = PROJECT_ROOT / 'outputs/phase1_1'
OUT_TABLES = OUT_ROOT / 'tables'
OUT_FIGS = OUT_ROOT / 'figures'
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

RESULT_TABLES = {}
LOG = []


def log(msg):
    print(msg)
    LOG.append(msg)


def save_df(name, df):
    RESULT_TABLES[name] = df.copy()


def fit_fe_controls(df_in, outcome):
    use_cols = ['Country Name', 'year', outcome, 'm2_growth', 'trade_open', 'gdp_pc_growth', 'pop_growth', 'investment_share']
    d = df_in[use_cols].dropna().copy()
    d = d.set_index(['Country Name', 'year']).sort_index()
    formula = f'{outcome} ~ 1 + m2_growth + trade_open + gdp_pc_growth + pop_growth + investment_share + EntityEffects + TimeEffects'
    model = PanelOLS.from_formula(formula, data=d)
    res = model.fit(cov_type='clustered', cluster_entity=True)
    return res, d


def res_to_df(res, model_name, outcome):
    rows = []
    for term in res.params.index:
        rows.append({
            'model': model_name,
            'outcome': outcome,
            'term': term,
            'coef': float(res.params.loc[term]),
            'std_error': float(res.std_errors.loc[term]),
            'p_value': float(res.pvalues.loc[term]),
            'nobs': int(res.nobs),
            'r2_within': float(res.rsquared_within),
            'r2_between': float(res.rsquared_between),
            'r2_overall': float(res.rsquared_overall),
        })
    return pd.DataFrame(rows)


In [ ]:
assert BASE_DATA.exists(), f'Missing: {BASE_DATA}'
assert CTRL_DATA.exists(), f'Missing: {CTRL_DATA}'
assert IV_DATA.exists(), f'Missing: {IV_DATA}'

base = pd.read_csv(BASE_DATA)
ctrl = pd.read_csv(CTRL_DATA)
iv = pd.read_csv(IV_DATA)

df = base.merge(ctrl, on=['Country Name', 'year'], how='left')
df = df.merge(iv, on=['Country Name', 'year'], how='left')

required = ['Country Name', 'year', 'm2_growth', 'inflation', 'gdp_growth', 'trade_open', 'gdp_pc_growth', 'pop_growth', 'investment_share', 'instrument_m2_l1', 'instrument_m2_external_level']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns after merge: {missing}')

summary = pd.DataFrame([
    {'metric': 'rows', 'value': len(df)},
    {'metric': 'countries', 'value': df['Country Name'].nunique()},
    {'metric': 'year_min', 'value': int(df['year'].min())},
    {'metric': 'year_max', 'value': int(df['year'].max())},
])
display(summary)

miss = df[required].isna().sum().to_frame('missing_count')
miss['missing_share'] = miss['missing_count'] / len(df)
display(miss)

save_df('phase1_1_data_summary.csv', summary)
save_df('phase1_1_missingness.csv', miss.reset_index().rename(columns={'index': 'column'}))
log('Data merge and contract checks completed.')


## FE + Controls Baseline

This section adds macro controls and keeps country + year fixed effects.


In [ ]:
fe_rows = []
for outcome in ['inflation', 'gdp_growth']:
    try:
        res, d_used = fit_fe_controls(df, outcome=outcome)
        out = res_to_df(res, model_name='fe_controls_twfe', outcome=outcome)
        fe_rows.append(out)
        log(f'FE+controls completed for {outcome}: nobs={int(res.nobs)}, coef_m2={float(res.params["m2_growth"]):.4f}, p={float(res.pvalues["m2_growth"]):.4g}')
    except Exception as e:
        log(f'FE+controls failed for {outcome}: {e}')

if fe_rows:
    fe_tbl = pd.concat(fe_rows, ignore_index=True)
    display(fe_tbl)
    save_df('phase1_1_fe_controls_results.csv', fe_tbl)


## First-Stage Diagnostics for IV Candidates

We test two candidates:
- `instrument_m2_l1`: lagged money growth (internal IV candidate).
- `instrument_m2_external_level`: external Bartik-style candidate (`depth_base × fed funds level`).

Diagnostics are shown under:
- Two-way FE first stage.
- Country-FE-only first stage.


In [ ]:
first_stage_rows = []

# two-way FE first-stage sample
fs_cols = ['m2_growth', 'trade_open', 'gdp_pc_growth', 'pop_growth', 'investment_share', 'Country Name', 'year', 'instrument_m2_l1', 'instrument_m2_external_level']
fs = df[fs_cols].dropna().rename(columns={'Country Name': 'country'}).copy()

for iv_col in ['instrument_m2_l1', 'instrument_m2_external_level']:
    # twfe
    twfe = smf.ols(
        f'm2_growth ~ {iv_col} + trade_open + gdp_pc_growth + pop_growth + investment_share + C(country) + C(year)',
        data=fs,
    ).fit()

    # country FE only
    cfe = smf.ols(
        f'm2_growth ~ {iv_col} + trade_open + gdp_pc_growth + pop_growth + investment_share + C(country)',
        data=fs,
    ).fit()

    first_stage_rows.append({
        'instrument': iv_col,
        'spec': 'twfe',
        'coef': float(twfe.params[iv_col]),
        't_stat': float(twfe.tvalues[iv_col]),
        'approx_F_t2': float(twfe.tvalues[iv_col] ** 2),
        'p_value': float(twfe.pvalues[iv_col]),
        'nobs': int(twfe.nobs),
    })
    first_stage_rows.append({
        'instrument': iv_col,
        'spec': 'country_fe_only',
        'coef': float(cfe.params[iv_col]),
        't_stat': float(cfe.tvalues[iv_col]),
        'approx_F_t2': float(cfe.tvalues[iv_col] ** 2),
        'p_value': float(cfe.pvalues[iv_col]),
        'nobs': int(cfe.nobs),
    })

first_stage_tbl = pd.DataFrame(first_stage_rows)
display(first_stage_tbl)
save_df('phase1_1_first_stage_diagnostics.csv', first_stage_tbl)

log('First-stage diagnostics completed.')


## 2SLS Spec A: Two-Way FE + Lagged Instrument

Second stage with country + year FE and controls.
Interpret this as a practical Phase 1.1 IV step, not final identification closure.


In [ ]:
iv_a_rows = []

a = df[['inflation', 'gdp_growth', 'm2_growth', 'instrument_m2_l1', 'trade_open', 'gdp_pc_growth', 'pop_growth', 'investment_share', 'Country Name', 'year']].dropna().copy()
a = a.rename(columns={'Country Name': 'country'})

for outcome in ['inflation', 'gdp_growth']:
    try:
        formula = f'{outcome} ~ 1 + trade_open + gdp_pc_growth + pop_growth + investment_share + C(country) + C(year) [m2_growth ~ instrument_m2_l1]'
        res = IV2SLS.from_formula(formula, data=a).fit(cov_type='clustered', clusters=a['country'])
        iv_a_rows.append({
            'model': 'iv_twfe_lag',
            'outcome': outcome,
            'coef_m2_growth': float(res.params['m2_growth']),
            'std_error_m2_growth': float(res.std_errors['m2_growth']),
            'p_value_m2_growth': float(res.pvalues['m2_growth']),
            'nobs': int(res.nobs),
            'r2': float(res.rsquared),
        })
        log(f'IV Spec A ({outcome}) complete: coef={float(res.params["m2_growth"]):.4f}, p={float(res.pvalues["m2_growth"]):.4g}, n={int(res.nobs)}')
    except Exception as e:
        log(f'IV Spec A failed for {outcome}: {e}')

if iv_a_rows:
    iv_a_tbl = pd.DataFrame(iv_a_rows)
    display(iv_a_tbl)
    save_df('phase1_1_iv_specA_twfe_lag.csv', iv_a_tbl)


## 2SLS Spec B: Country FE + External Bartik Instrument

This checks the external IV candidate in a less restrictive FE setup.
Caveat: dropping year FE increases risk of omitted global-time shocks.


In [ ]:
iv_b_rows = []

b = df[['inflation', 'gdp_growth', 'm2_growth', 'instrument_m2_external_level', 'trade_open', 'gdp_pc_growth', 'pop_growth', 'investment_share', 'Country Name', 'year']].dropna().copy()
b = b.rename(columns={'Country Name': 'country'})

for outcome in ['inflation', 'gdp_growth']:
    try:
        formula = f'{outcome} ~ 1 + trade_open + gdp_pc_growth + pop_growth + investment_share + C(country) [m2_growth ~ instrument_m2_external_level]'
        res = IV2SLS.from_formula(formula, data=b).fit(cov_type='clustered', clusters=b['country'])
        iv_b_rows.append({
            'model': 'iv_countryfe_external',
            'outcome': outcome,
            'coef_m2_growth': float(res.params['m2_growth']),
            'std_error_m2_growth': float(res.std_errors['m2_growth']),
            'p_value_m2_growth': float(res.pvalues['m2_growth']),
            'nobs': int(res.nobs),
            'r2': float(res.rsquared),
        })
        log(f'IV Spec B ({outcome}) complete: coef={float(res.params["m2_growth"]):.4f}, p={float(res.pvalues["m2_growth"]):.4g}, n={int(res.nobs)}')
    except Exception as e:
        log(f'IV Spec B failed for {outcome}: {e}')

if iv_b_rows:
    iv_b_tbl = pd.DataFrame(iv_b_rows)
    display(iv_b_tbl)
    save_df('phase1_1_iv_specB_countryfe_external.csv', iv_b_tbl)


## 2SLS Spec C: Two-Way FE + External Bartik Instrument

This runs the external candidate under the stricter country + year FE setup.


In [ ]:
iv_c_rows = []

c = df[['inflation', 'gdp_growth', 'm2_growth', 'instrument_m2_external_level', 'trade_open', 'gdp_pc_growth', 'pop_growth', 'investment_share', 'Country Name', 'year']].dropna().copy()
c = c.rename(columns={'Country Name': 'country'})

for outcome in ['inflation', 'gdp_growth']:
    try:
        formula = f'{outcome} ~ 1 + trade_open + gdp_pc_growth + pop_growth + investment_share + C(country) + C(year) [m2_growth ~ instrument_m2_external_level]'
        res = IV2SLS.from_formula(formula, data=c).fit(cov_type='clustered', clusters=c['country'])
        iv_c_rows.append({
            'model': 'iv_twfe_external',
            'outcome': outcome,
            'coef_m2_growth': float(res.params['m2_growth']),
            'std_error_m2_growth': float(res.std_errors['m2_growth']),
            'p_value_m2_growth': float(res.pvalues['m2_growth']),
            'nobs': int(res.nobs),
            'r2': float(res.rsquared),
        })
        log(f'IV Spec C ({outcome}) complete: coef={float(res.params["m2_growth"]):.4f}, p={float(res.pvalues["m2_growth"]):.4g}, n={int(res.nobs)}')
    except Exception as e:
        log(f'IV Spec C failed for {outcome}: {e}')

if iv_c_rows:
    iv_c_tbl = pd.DataFrame(iv_c_rows)
    display(iv_c_tbl)
    save_df('phase1_1_iv_specC_twfe_external.csv', iv_c_tbl)


## Evaluation: What We Did and What Holds

This section consolidates core evidence from Phase 1.1 and gives a practical decision readout.


In [ ]:
# collect comparable effect rows
eval_rows = []

if 'phase1_1_fe_controls_results.csv' in RESULT_TABLES:
    fe = RESULT_TABLES['phase1_1_fe_controls_results.csv']
    fe_m2 = fe[fe['term'] == 'm2_growth'][['model', 'outcome', 'coef', 'std_error', 'p_value', 'nobs']].copy()
    fe_m2 = fe_m2.rename(columns={'coef': 'coef_m2_growth', 'std_error': 'std_error_m2_growth', 'p_value': 'p_value_m2_growth'})
    eval_rows.append(fe_m2)

if 'phase1_1_iv_specA_twfe_lag.csv' in RESULT_TABLES:
    eval_rows.append(RESULT_TABLES['phase1_1_iv_specA_twfe_lag.csv'][['model', 'outcome', 'coef_m2_growth', 'std_error_m2_growth', 'p_value_m2_growth', 'nobs']])

if 'phase1_1_iv_specB_countryfe_external.csv' in RESULT_TABLES:
    eval_rows.append(RESULT_TABLES['phase1_1_iv_specB_countryfe_external.csv'][['model', 'outcome', 'coef_m2_growth', 'std_error_m2_growth', 'p_value_m2_growth', 'nobs']])

if 'phase1_1_iv_specC_twfe_external.csv' in RESULT_TABLES:
    eval_rows.append(RESULT_TABLES['phase1_1_iv_specC_twfe_external.csv'][['model', 'outcome', 'coef_m2_growth', 'std_error_m2_growth', 'p_value_m2_growth', 'nobs']])

if eval_rows:
    eval_tbl = pd.concat(eval_rows, ignore_index=True)
    display(eval_tbl)
    save_df('phase1_1_model_comparison.csv', eval_tbl)

# quick first-stage signal chart
if 'phase1_1_first_stage_diagnostics.csv' in RESULT_TABLES:
    fst = RESULT_TABLES['phase1_1_first_stage_diagnostics.csv'].copy()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=fst, x='instrument', y='approx_F_t2', hue='spec', ax=ax)
    ax.axhline(10, color='red', linestyle='--', label='Rule-of-thumb F=10')
    ax.set_title('First-Stage Strength Diagnostics')
    ax.set_ylabel('Approximate F-stat (t^2)')
    ax.legend()
    plt.show()
    fig.savefig(OUT_FIGS / 'phase1_1_first_stage_strength.png', dpi=160, bbox_inches='tight')


## Decision Readout (Phase 1.1)

Use this as the operational interpretation:
- If the two-way FE + lag-IV result stays stable and robustness is coherent, you have a defensible Phase 1.1 portfolio artifact.
- Treat the external country-FE IV as supplementary evidence only (weaker identification defense without year FE).
- Next highest-value move: add one carefully justified instrument upgrade or one high-quality control expansion, then freeze scope.


In [ ]:
# write all tables
written = []
for name, df_out in RESULT_TABLES.items():
    out_path = OUT_TABLES / name
    df_out.to_csv(out_path, index=False)
    written.append(str(out_path))

# write log
log_path = OUT_ROOT / 'phase1_1_log.md'
log_path.write_text('\n'.join(['# Phase 1.1 Log', ''] + [f'- {x}' for x in LOG]))

# simple evaluation markdown
eval_md = OUT_ROOT / 'phase1_1_evaluation.md'
lines = [
    '# Phase 1.1 Evaluation',
    '',
    '## What was added',
    '- Controls dataset from World Bank API',
    '- Instrument dataset with lag and external candidates',
    '- FE+controls + IV second-stage estimates',
    '',
    '## Key caution',
    '- Prioritize specifications with strong first-stage diagnostics and explicit identification caveats.',
]
eval_md.write_text('\n'.join(lines))

manifest = pd.DataFrame({'artifact_path': written + [str(log_path), str(eval_md)]})
manifest_path = OUT_ROOT / 'phase1_1_manifest.csv'
manifest.to_csv(manifest_path, index=False)

print('Tables written:', len(written))
print('Log:', log_path)
print('Evaluation:', eval_md)
print('Manifest:', manifest_path)
